# Concept 4 (Advanced): Validation & Retry Loops

A schema doesn't guarantee a valid response. Models can still return malformed JSON, the wrong type, or even call the wrong tool. This notebook builds a validate-and-retry loop that turns 'worked once' into 'works reliably'.

## Common Reliability Techniques
- **Schema validation:** check every response before trusting it
- **Retry with feedback:** send the validation error back to the model and ask again
- **Lower temperature:** more deterministic, less creative drift on structured calls
- **Cap retries:** fail loudly after N attempts instead of looping forever

In [ ]:
!pip install -q jsonschema

## Demo: A Validate-and-Retry Loop
`call_model_fn` simulates a model that sometimes returns malformed JSON on the first try, then self-corrects when told what was wrong. Swap it for a real API call in the hands-on task.

In [ ]:
import json

from jsonschema import validate, ValidationError



schema = {

    "type": "object",

    "properties": {

        "city": {"type": "string"},

        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}

    },

    "required": ["city", "unit"]

}



def simulated_model_call(error_feedback, attempt_state={"n": 0}):

    """Stands in for a real LLM call: fails once, then returns valid JSON."""

    attempt_state["n"] += 1

    if attempt_state["n"] == 1:

        return '{"city": "Pune"}'  # missing required "unit"

    return '{"city": "Pune", "unit": "celsius"}'



def get_valid_arguments(call_model_fn, schema, max_retries=3):

    error_feedback = None

    for attempt in range(1, max_retries + 1):

        raw = call_model_fn(error_feedback)

        try:

            args = json.loads(raw)

            validate(instance=args, schema=schema)

            return args, attempt

        except (json.JSONDecodeError, ValidationError) as e:

            error_feedback = f"Your last response was invalid: {e}. Return valid JSON matching the schema."

            print(f"Attempt {attempt} failed: {error_feedback}")

    raise RuntimeError("Model failed to return valid arguments after retries")



args, attempts = get_valid_arguments(simulated_model_call, schema)

print(f"Succeeded on attempt {attempts}:", args)

## Try It Yourself
1. Replace `simulated_model_call` with a real OpenAI tool call (from Concept 3) that re-sends `error_feedback` as an extra message when retrying

2. Lower `max_retries` to 1 and see the `RuntimeError` fire